In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import polars as pl
from pathlib import Path

from src.training.splits import make_backtest_folds, get_fold_data
from src.training.dataset import get_feature_cols, CAT_FEATURES, TARGET
from src.training.metrics import compute_fold_weights_and_scales, calculate_sampled_wrmsse
import lightgbm as lgb

FEATURES_DIR = Path("../data/processed/features")
MODELS_DIR = Path("../models")

In [2]:
model = lgb.Booster(model_file=str(MODELS_DIR / "lgbm_fold_3.txt"))

lf = pl.scan_parquet(FEATURES_DIR / "*.parquet")
dates = lf.select("date").collect()["date"].unique().sort().to_list()
folds = make_backtest_folds(dates)
fold3 = folds[2]
print(fold3)

_, val_lf = get_fold_data(lf, fold3)
val_df = val_lf.collect()

feature_cols = get_feature_cols(val_df.columns)

for col in CAT_FEATURES:
    val_df = val_df.with_columns(
        pl.col(col).cast(pl.Categorical).to_physical().cast(pl.Int32)
    )

X_val = np.asarray(val_df.select(feature_cols).to_numpy(), dtype=np.float32)
y_val = val_df.select(TARGET).to_numpy().ravel()
val_ids = val_df.select("id").to_numpy().ravel()
val_dates = val_df.select("date").to_numpy().ravel()

y_pred = np.clip(model.predict(X_val), 0, None)
POST_PROCESSING_MULTIPLIER = 0.975
y_pred = y_pred * POST_PROCESSING_MULTIPLIER

print(f"Val shape: {X_val.shape}")
print(f"y_val range: [{y_val.min()}, {y_val.max()}]")
print(f"y_pred range: [{y_pred.min():.4f}, {y_pred.max():.4f}]")

Fold 3: train [2011-01-29 : 2016-03-27] gap 28d val [2016-04-25 : 2016-05-22]
Val shape: (853720, 50)
y_val range: [0, 196]
y_pred range: [0.0057, 166.7661]


In [3]:
df_weight_ref = pl.scan_parquet(FEATURES_DIR / "*.parquet").select(
    ["id", "date", "sales", "sell_price"]
).collect()

m5_ref = compute_fold_weights_and_scales(df_weight_ref, fold3.train_end)

print(f"Reference series: {len(m5_ref):,}")
weights = [v['weight'] for v in m5_ref.values()]
scales = [v['scale'] for v in m5_ref.values()]
print(f"Weights sum: {sum(weights):.6f}")
print(f"Zero weights: {sum(1 for w in weights if w == 0):,}")
print(f"Scale range: [{min(scales):.4f}, {max(scales):.4f}]")

Reference series: 30,490
Weights sum: 1.000000
Zero weights: 2,149
Scale range: [0.0015, 4971.0902]


In [4]:
sample = val_df.filter(pl.col("id") == "FOODS_3_090_CA_1_evaluation").sort("date")
corr_dept = sample.select(pl.corr("sales", "item_share_dept")).item()
corr_cat = sample.select(pl.corr("sales", "item_share_cat")).item()
print(f"Correlation sales vs item_share_dept: {corr_dept:.4f}")
print(f"Correlation sales vs item_share_cat:  {corr_cat:.4f}")
print("\nExpect ~0.3-0.6 if no leakage, >0.8 indicates leakage")
print(sample.select(["date", "sales", "item_share_dept", "item_share_cat"]).head(10))

Correlation sales vs item_share_dept: 0.5126
Correlation sales vs item_share_cat:  0.5255

Expect ~0.3-0.6 if no leakage, >0.8 indicates leakage
shape: (10, 4)
┌────────────┬───────┬─────────────────┬────────────────┐
│ date       ┆ sales ┆ item_share_dept ┆ item_share_cat │
│ ---        ┆ ---   ┆ ---             ┆ ---            │
│ date       ┆ i32   ┆ f32             ┆ f32            │
╞════════════╪═══════╪═════════════════╪════════════════╡
│ 2016-04-25 ┆ 48    ┆ 0.014047        ┆ 0.010363       │
│ 2016-04-26 ┆ 35    ┆ 0.022419        ┆ 0.016592       │
│ 2016-04-27 ┆ 34    ┆ 0.019898        ┆ 0.0139         │
│ 2016-04-28 ┆ 67    ┆ 0.019026        ┆ 0.013782       │
│ 2016-04-29 ┆ 63    ┆ 0.038999        ┆ 0.027392       │
│ 2016-04-30 ┆ 99    ┆ 0.028873        ┆ 0.020778       │
│ 2016-05-01 ┆ 71    ┆ 0.034835        ┆ 0.025063       │
│ 2016-05-02 ┆ 59    ┆ 0.022676        ┆ 0.01669        │
│ 2016-05-03 ┆ 35    ┆ 0.024624        ┆ 0.017999       │
│ 2016-05-04 ┆ 47    ┆ 0.016

In [5]:
corr_lag1 = sample.select(pl.corr("sales", "lag_1")).item()
corr_roll7 = sample.select(pl.corr("sales", "roll_mean_7")).item()
print(f"Correlation sales vs lag_1:      {corr_lag1:.4f}  (expect ~0.7)")
print(f"Correlation sales vs roll_mean_7: {corr_roll7:.4f}  (expect ~0.8, higher due to smoothing)")
print("\nIf roll_mean_7 > 0.95 there is likely leakage")
print(sample.select(["date", "sales", "lag_1", "roll_mean_7"]).head(10))

Correlation sales vs lag_1:      0.4541  (expect ~0.7)
Correlation sales vs roll_mean_7: 0.0822  (expect ~0.8, higher due to smoothing)

If roll_mean_7 > 0.95 there is likely leakage
shape: (10, 4)
┌────────────┬───────┬───────┬─────────────┐
│ date       ┆ sales ┆ lag_1 ┆ roll_mean_7 │
│ ---        ┆ ---   ┆ ---   ┆ ---         │
│ date       ┆ i32   ┆ i32   ┆ f64         │
╞════════════╪═══════╪═══════╪═════════════╡
│ 2016-04-25 ┆ 48    ┆ 42    ┆ 54.428571   │
│ 2016-04-26 ┆ 35    ┆ 48    ┆ 57.0        │
│ 2016-04-27 ┆ 34    ┆ 35    ┆ 55.571429   │
│ 2016-04-28 ┆ 67    ┆ 34    ┆ 56.285714   │
│ 2016-04-29 ┆ 63    ┆ 67    ┆ 58.285714   │
│ 2016-04-30 ┆ 99    ┆ 63    ┆ 54.857143   │
│ 2016-05-01 ┆ 71    ┆ 99    ┆ 55.428571   │
│ 2016-05-02 ┆ 59    ┆ 71    ┆ 59.571429   │
│ 2016-05-03 ┆ 35    ┆ 59    ┆ 61.142857   │
│ 2016-05-04 ┆ 47    ┆ 35    ┆ 61.142857   │
└────────────┴───────┴───────┴─────────────┘


In [6]:
train_lf, val_lf_check = get_fold_data(lf, fold3)
train_dates = set(train_lf.select("date").collect()["date"].unique().to_list())
val_dates_set = set(val_lf_check.select("date").collect()["date"].unique().to_list())
overlap = train_dates & val_dates_set

print(f"Train date range: {min(train_dates)} to {max(train_dates)}")
print(f"Val date range:   {min(val_dates_set)} to {max(val_dates_set)}")
print(f"Overlapping dates: {len(overlap)}")
print(f"Gap days: {(min(val_dates_set) - max(train_dates)).days}")

Train date range: 2011-01-29 to 2016-03-27
Val date range:   2016-04-25 to 2016-05-22
Overlapping dates: 0
Gap days: 29


In [7]:
# naive forecast: use lag_1 (yesterday's sales) as prediction
lag1_idx = feature_cols.index("lag_1")
naive_pred = np.clip(X_val[:, lag1_idx], 0, None)

# seasonal naive: use lag_7 (same day last week)
lag7_idx = feature_cols.index("lag_7")
seasonal_naive_pred = np.clip(X_val[:, lag7_idx], 0, None)

naive_wrmsse = calculate_sampled_wrmsse(y_val, naive_pred, val_ids, m5_ref)
seasonal_naive_wrmsse = calculate_sampled_wrmsse(y_val, seasonal_naive_pred, val_ids, m5_ref)
model_wrmsse = calculate_sampled_wrmsse(y_val, y_pred, val_ids, m5_ref)

print(f"Naive lag_1 WRMSSE:        {naive_wrmsse:.4f}  (expect ~1.0 by definition)")
print(f"Seasonal naive lag_7 WRMSSE: {seasonal_naive_wrmsse:.4f}  (expect ~0.8-1.0)")
print(f"Model WRMSSE:              {model_wrmsse:.4f}")
print()
print("If naive lag_1 is not ~1.0, the scale computation is wrong")
print("If model beats naive by 5x+, there is likely leakage")

Naive lag_1 WRMSSE:        1.0665  (expect ~1.0 by definition)
Seasonal naive lag_7 WRMSSE: 1.0755  (expect ~0.8-1.0)
Model WRMSSE:              0.7838

If naive lag_1 is not ~1.0, the scale computation is wrong
If model beats naive by 5x+, there is likely leakage


In [8]:
zero_pred = np.zeros_like(y_val)
zero_wrmsse = calculate_sampled_wrmsse(y_val, zero_pred, val_ids, m5_ref)
mean_pred = np.full_like(y_val, y_val.mean())
mean_wrmsse = calculate_sampled_wrmsse(y_val, mean_pred, val_ids, m5_ref)

print(f"Zero prediction WRMSSE: {zero_wrmsse:.4f}")
print(f"Mean prediction WRMSSE: {mean_wrmsse:.4f}")
print(f"Model WRMSSE:           {model_wrmsse:.4f}")
print()
print("Zero prediction should be worse than naive")
print("If model is close to zero prediction, something is very wrong")

Zero prediction WRMSSE: 1.3798
Mean prediction WRMSSE: 1.2459
Model WRMSSE:           0.7838

Zero prediction should be worse than naive
If model is close to zero prediction, something is very wrong


In [9]:
# correlation on full val set, not just one series
val_df_check = pl.scan_parquet("../data/processed/features/*.parquet").filter(
    (pl.col("date") >= pl.date(2016, 4, 25)) &
    (pl.col("date") <= pl.col("date").max())
).collect()

corr_roll7 = val_df_check.select(pl.corr("sales", "roll_mean_7")).item()
corr_lag1 = val_df_check.select(pl.corr("sales", "lag_1")).item()
corr_share_dept = val_df_check.select(pl.corr("sales", "item_share_dept")).item()
corr_share_cat = val_df_check.select(pl.corr("sales", "item_share_cat")).item()

print("Full val set correlations:")
print(f"roll_mean_7:    {corr_roll7:.4f}  (expect ~0.7-0.8)")
print(f"lag_1:          {corr_lag1:.4f}  (expect ~0.6-0.7)")
print(f"item_share_dept:{corr_share_dept:.4f}  (expect ~0.3-0.5, no leakage)")
print(f"item_share_cat: {corr_share_cat:.4f}  (expect ~0.3-0.5, no leakage)")

Full val set correlations:
roll_mean_7:    0.8177  (expect ~0.7-0.8)
lag_1:          0.7390  (expect ~0.6-0.7)
item_share_dept:0.3000  (expect ~0.3-0.5, no leakage)
item_share_cat: 0.4003  (expect ~0.3-0.5, no leakage)
